In [1]:
import mne
import os
import numpy as np

c:\Users\jackr\anaconda3\envs\VR\Lib\site-packages\mne\datasets\eegbci\eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [19]:
# Put path to your data folder here
username = os.getlogin()
gender = "Women"
type = "OnField"  # "OnField" or "VR"
edf_folder = f'C:\\Users\\{username}\\Box\\Research_SPORTS_VR_AI\\Creative Inquiry F24\\EDF Data\\{gender}\\{type}'


In [20]:
# Define parameters
tmin = -2.5  # start of epoch relative to event
tmax = 2.5   # end of epoch relative to event
baseline = (None, 0)  # optional: baseline correction

# Output directory
output_dir = f'{gender}_{type}_epochs_as_edf_by_annotation'
os.makedirs(output_dir, exist_ok=True)



In [21]:
# Go through each file in the folder
for filename in os.listdir(edf_folder):
    # Check if the file is an EDF file
    if filename.endswith('.edf'):
        # Load the EDF file
        edf_path = os.path.join(edf_folder, filename)
        try:
            raw = mne.io.read_raw_edf(edf_path, preload=True)
        except:
            print(f'Error loading {filename}, skipping.')
            continue    
        print(f'Processing file: {filename}')

        # Take out the name, type, and number(s) of the file
        name, type, number = filename.split('_')
        number = number.split('.')[0]  # Remove file extension
        # Extract events and event IDs from annotations
        events, event_id = mne.events_from_annotations(raw)

        # Create epochs from raw data
        try:
            epochs = mne.Epochs(raw, events, event_id, tmin=tmin, tmax=tmax,
                    baseline=baseline, preload=True)
        except: 
            print(f'Error creating epochs for {filename}, skipping.')
            continue

        sfreq = raw.info['sfreq']

        print(f"Processing {len(epochs)} epochs...")
        print("Preserving original annotation timings within each epoch.\n")

        for i in range(len(epochs)):
            # Extract epoch data and info
            single_epoch = epochs[i]
            data = single_epoch.get_data().squeeze()
            info = single_epoch.info

            # Ensure 2D data shape
            if data.ndim == 1:
                data = data.reshape(1, -1)

            # Get absolute event time for this epoch
            event_sample = epochs.events[i, 0]
            event_time = event_sample / sfreq  # in seconds
            epoch_start_time = event_time + tmin
            epoch_end_time = event_time + tmax

            # Filter annotations that fall within this epoch window
            anns_in_epoch = []
            for onset, duration, description in zip(raw.annotations.onset,
                                                    raw.annotations.duration,
                                                    raw.annotations.description):
                if epoch_start_time <= onset <= epoch_end_time:
                    # Calculate relative onset time within the epoch
                    relative_onset = onset - epoch_start_time
                    anns_in_epoch.append((relative_onset, duration, description))
            
            # Create a RawArray for this epoch
            raw_epoch = mne.io.RawArray(data, info)

            # Add annotations to RawArray
            if anns_in_epoch:
                ann_onsets, ann_durations, ann_descs = zip(*anns_in_epoch)
                annotations = mne.Annotations(onset=ann_onsets,
                                            duration=ann_durations,
                                            description=ann_descs)
                raw_epoch.set_annotations(annotations)
            else:
                raw_epoch.set_annotations(mne.Annotations([], [], []))

            # Save the epoch to EDF
            filename = f'{name}_{type}_{number}_epoch_{i+1:03d}.edf'
            output_path = os.path.join(output_dir, filename)

            try:
                mne.export.export_raw(output_path, raw_epoch, overwrite=True)
                print(f"Epoch {i+1}: Saved with {len(anns_in_epoch)} annotation(s)")
            except Exception as e:
                print(f"Error saving epoch {i+1}: {e}")

            
print(f"\n✓ Complete! Saved {len(epochs)} epochs to '{output_dir}'")
print("Each epoch includes only annotations that occurred within its time window.\n")



Extracting EDF parameters from C:\Users\jackr\Box\Research_SPORTS_VR_AI\Creative Inquiry F24\EDF Data\Women\OnField\Jenna_OnField_1-0.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 17999  =      0.000 ...    59.997 secs...
Processing file: Jenna_OnField_1-0.edf
Used Annotations descriptions: [np.str_(' Pass ')]
Not setting metadata
5 matching events found
Setting baseline interval to [-2.5, 0.0] sec
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 5 events and 1501 original time points ...
0 bad epochs dropped
Processing 5 epochs...
Preserving original annotation timings within each epoch.

Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.
Epoch 1: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.

C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)


Epoch 4: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.
Epoch 5: Saved with 1 annotation(s)
Extracting EDF parameters from C:\Users\jackr\Box\Research_SPORTS_VR_AI\Creative Inquiry F24\EDF Data\Women\OnField\Jenna_OnField_1-1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 25799  =      0.000 ...    85.997 secs...
Processing file: Jenna_OnField_1-1.edf
Error creating epochs for Jenna_OnField_1-1.edf, skipping.
Extracting EDF parameters from C:\Users\jackr\Box\Research_SPORTS_VR_AI\Creative Inquiry F24\EDF Data\Women\OnField\Jenna_OnField_2-0.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 86999  =      0.000 ...   289.997 secs...


C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)


Processing file: Jenna_OnField_2-0.edf
Used Annotations descriptions: [np.str_(' Pass ')]
Not setting metadata
14 matching events found
Setting baseline interval to [-2.5, 0.0] sec
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 14 events and 1501 original time points ...
0 bad epochs dropped
Processing 14 epochs...
Preserving original annotation timings within each epoch.

Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.
Epoch 1: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.
Epoch 2: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.


...

Epoch 7: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.
Epoch 8: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.
Epoch 9: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.
Epoch 10: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.


C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blo

Epoch 11: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.
Epoch 12: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.
Epoch 13: Saved with 1 annotation(s)
Creating RawArray with float64 data, n_channels=30, n_times=1501
    Range : 0 ... 1500 =      0.000 ...     5.000 secs
Ready.
Epoch 14: Saved with 1 annotation(s)
Extracting EDF parameters from C:\Users\jackr\Box\Research_SPORTS_VR_AI\Creative Inquiry F24\EDF Data\Women\OnField\Neely_OnField_4-1.edf...
EDF file detected


C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blocks, so 0.9966666666666667 seconds of zeros were appended to all channels when writing the final block.
  mne.export.export_raw(output_path, raw_epoch, overwrite=True)
C:\Users\jackr\AppData\Local\Temp\ipykernel_54576\587981652.py:77: RuntimeWarning: EDF format requires equal-length data blo

Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 22199  =      0.000 ...    73.997 secs...
Processing file: Neely_OnField_4-1.edf
Error creating epochs for Neely_OnField_4-1.edf, skipping.

✓ Complete! Saved 14 epochs to 'Women_OnField_epochs_as_edf_by_annotation'
Each epoch includes only annotations that occurred within its time window.

